# Identify driver factors in cell state transitions

The ``predict_transition_driver_regulators`` command identifies key transcription factors that drive changes in gene expression and/or chromatin accessibility during cell state transitions (e.g., differentiation or reprogramming).

You can run this command with:
- expression only,
- accessibility only, or
- both expression and accessibility.

Provide the corresponding input files for the analyses you want to perform.

**Note**: The remaining examples will only show the direct command usage. 

If you need to use Singularity container, please refer to the [`singularity_use.ipynb`](singularity_use.ipynb) tutorial for detailed instructions on using `singularity exec` with `chrombert-tools`.

## Example:

Find driver regulators in fibroblast-to-myoblast transition using both expression and accessibility


In [1]:
import pandas as pd
import numpy as np
import os


In [2]:
os.environ["CUDA_VISIBLE_DEVICES"]='1'

In [3]:
!chrombert-tools -h

Usage: chrombert-tools [OPTIONS] COMMAND [ARGS]...

  Type -h or --help after any subcommand for more information.

Options:
  -v, --verbose  Verbose logging
  -d, --debug    Post mortem debugging
  -V, --version  Show the version and exit.
  -h, --help     Show this message and exit.

Commands:
  embed_region                    Generate region...
  embed_regulator                 Extract regulator...
  gene_activity_repression        Predict gene...
  interpret_region_region_interactions
                                  Region embedding...
  interpret_regulator_effects_between_region_groups
                                  Identify regulators...
  interpret_regulator_regulator_interactions
                                  Interpret...
  predict_cell_type_master_regulators
                                  Find candidate key...
  predict_regulator_context_cofactors
                                  Identify...
  predict_tf_binding_regions      Predict TF binding...
  predict_transit

In [4]:
!chrombert-tools predict_transition_driver_regulators -h

Usage: chrombert-tools predict_transition_driver_regulators 
           [OPTIONS]

  Find driver factors in cell state transitions.

  Provide at least one of: - Expression: --exp-tpm1 and --exp-tpm2 -
  Accessibility: --acc-peak1 and --acc-signal1 (optional --acc-peak2; add
  --acc-signal2 for fold-change / acc stage 3).

  Merged rankings require both expression and two-state accessibility
  rankings.

Options:
  --exp-tpm1 FILE                 Expression (TPM) file for cell state 1. CSV
                                  format with 'gene_id' and 'tpm' columns.
  --exp-tpm2 FILE                 Expression (TPM) file for cell state 2. CSV
                                  format with 'gene_id' and 'tpm' columns.
  --acc-peak1 TEXT                BED peak file(s) for state 1; use ';' for
                                  multiple.
  --acc-peak2 TEXT                Optional BED peak file(s) for state 2; use
                                  ';' for multiple.
  --acc-signal1 TEXT        

## Run

In [5]:
# Download data
import subprocess
if not os.path.exists('../data/myoblast_ENCFF647RNC_peak.bed'):
    cmd = f'wget https://www.encodeproject.org/files/ENCFF647RNC/@@download/ENCFF647RNC.bed.gz -O ../data/myoblast_ENCFF647RNC_peak.bed.gz'
    subprocess.run(cmd, shell=True)
    cmd = f"gzip -d ../data/myoblast_ENCFF647RNC_peak.bed.gz"
    subprocess.run(cmd, shell=True)

if not os.path.exists('../data/myoblast_ENCFF149ERN_signal.bigwig'):
    cmd = f'wget https://www.encodeproject.org/files/ENCFF149ERN/@@download/ENCFF149ERN.bigWig -O ../data/myoblast_ENCFF149ERN_signal.bigwig'
    subprocess.run(cmd, shell=True) 


if not os.path.exists('../data/fibroblast_ENCFF184KAM_peak.bed'):
    cmd = f'wget https://www.encodeproject.org/files/ENCFF184KAM/@@download/ENCFF184KAM.bed.gz -O ../data/fibroblast_ENCFF184KAM_peak.bed.gz'
    subprocess.run(cmd, shell=True)
    cmd = f"gzip -d ../data/fibroblast_ENCFF184KAM_peak.bed.gz"
    subprocess.run(cmd, shell=True)


if not os.path.exists('../data/fibroblast_ENCFF361BTT_signal.bigwig'):
    cmd = 'wget https://www.encodeproject.org/files/ENCFF361BTT/@@download/ENCFF361BTT.bigWig -O ../data/fibroblast_ENCFF361BTT_signal.bigwig'
    subprocess.run(cmd, shell=True)

In [6]:
# Runtime estimates:
#   - fast mode: ~3-5 hours
#     (uses all ~19,620 genes for expression analysis, but downsamples 
#      chromatin accessibility regions to 20k for faster training)
#
# Note: Both modes (fast and full) use the complete gene expression dataset. The 'fast' mode 
# only downsamples chromatin accessibility regions, not gene data.

# So this downsampled 5000 genes for expression analysis for test (40-100 minutes)

!chrombert-tools predict_transition_driver_regulators \
  --exp-tpm1 "../data/fibroblast_expression_sample5000.csv" \
  --exp-tpm2 "../data/myoblast_expression_sample5000.csv" \
  --acc-peak1 "../data/fibroblast_ENCFF184KAM_peak.bed" \
  --acc-peak2 "../data/myoblast_ENCFF647RNC_peak.bed" \
  --acc-signal1 "../data/fibroblast_ENCFF361BTT_signal.bigwig" \
  --acc-signal2 "../data/myoblast_ENCFF149ERN_signal.bigwig" \
  --genome 'hg38' \
  --resolution '1kb' \
  --odir output_find_driver_in_transition \
  --direction "2-1" 2> "./tmp/hg38_1kb.stderr.log"

Whether to train ChromBERT to predict chromatin accessibility changes in cell state transition: True
Stage 1 (acc): prepare dataset
Processing stage 1: prepare chromatin accessibility dataset
  Mode: two states (fold-change label)
  TSS ± flank background regions: enabled, flank distance: 10000 bp
[state1_0] Region summary - total: 287892, overlapping with ChromBERT: 284160 (one region may overlap multiple ChromBERT regions, we keep overlaps with ≥50% coverage of either the ChromBERT bin or the input region), non-overlapping: 5967
  [state1] merged 1 peak file(s) → 250143 unique ChromBERT regions (after overlap + dedup)
[state2_0] Region summary - total: 373422, overlapping with ChromBERT: 368260 (one region may overlap multiple ChromBERT regions, we keep overlaps with ≥50% coverage of either the ChromBERT bin or the input region), non-overlapping: 7920
  [state2] merged 1 peak file(s) → 324690 unique ChromBERT regions (after overlap + dedup)
[tss_bg] Region summary - total: 19260, ove

In [ ]:
# Runtime estimates:
#   - fast mode: ~3-5 hours
#     (uses all ~19,620 genes for expression analysis, but downsamples 
#      chromatin accessibility regions to 20k for faster training)
#
# Note: Both modes (fast and full) use the complete gene expression dataset. The 'fast' mode 
# only downsamples chromatin accessibility regions, not gene data.

# So this downsampled 5000 genes for expression analysis for test (40-100 minutes)

!chrombert-tools predict_transition_driver_regulators \
  --exp-tpm1 "../data/fibroblast_expression_sample5000.csv" \
  --exp-tpm2 "../data/myoblast_expression_sample5000.csv" \
  --acc-peak1 "../data/fibroblast_ENCFF184KAM_peak.bed" \
  --acc-peak2 "../data/myoblast_ENCFF647RNC_peak.bed" \
  --acc-signal1 "../data/fibroblast_ENCFF361BTT_signal.bigwig" \
  --acc-signal2 "../data/myoblast_ENCFF149ERN_signal.bigwig" \
  --ft-ckpt-acc "/mnt/Storage2/home/chenqianqian/projects/chrombert/chrombert_tools/ChromBERT-tools/examples/cli/output_find_driver_in_transition/acc/train/try_00_seed_55/lightning_logs/lightning_logs/version_0/checkpoints/epoch=2-step=163.ckpt" \
  --ft-ckpt-exp "/mnt/Storage2/home/chenqianqian/projects/chrombert/chrombert_tools/ChromBERT-tools/examples/cli/output_find_driver_in_transition/exp/train/try_00_seed_55/lightning_logs/lightning_logs/version_0/checkpoints/epoch=2-step=163.ckpt" \
  --genome 'hg38' \
  --resolution '1kb' \
  --odir output_find_driver_in_transition \
  --direction "2-1" 2> "./tmp/hg38_1kb.stderr.log"

Whether to train ChromBERT to predict chromatin accessibility changes in cell state transition: True
Stage 1 (acc): prepare dataset
Chromatin accessibility dataset already exists in output_find_driver_in_transition/acc/dataset
Finished Stage 1 (acc)
Stage 2 (acc): train ChromBERT to predict chromatin accessibility changes in cell state transition
Using fine-tuned checkpoint: /mnt/Storage2/home/chenqianqian/projects/chrombert/chrombert_tools/ChromBERT-tools/examples/cli/output_find_driver_in_transition/acc/train/try_00_seed_55/lightning_logs/lightning_logs/version_0/checkpoints/epoch=2-step=163.ckpt
Load pretrained ckpt /mnt/Storage/home/chenqianqian/.cache/chrombert/data/checkpoint/hg38_6k_1kb_pretrain.ckpt successfully!
Loading checkpoint from /mnt/Storage2/home/chenqianqian/projects/chrombert/chrombert_tools/ChromBERT-tools/examples/cli/output_find_driver_in_transition/acc/train/try_00_seed_55/lightning_logs/lightning_logs/version_0/checkpoints/epoch=2-step=163.ckpt
Loading from pl m

In [13]:
!chrombert-tools predict_transition_driver_regulators \
  --exp-tpm1 "../data/fibroblast_expression_sample5000.csv" \
  --exp-tpm2 "../data/myoblast_expression_sample5000.csv" \
  --acc-peak1 "../data/fibroblast_ENCFF184KAM_peak.bed" \
  --acc-peak2 "../data/myoblast_ENCFF647RNC_peak.bed" \
  --acc-signal1 "../data/fibroblast_ENCFF361BTT_signal.bigwig" \
  --acc-signal2 "../data/myoblast_ENCFF149ERN_signal.bigwig" \
  --ft-ckpt-acc "/mnt/Storage2/home/chenqianqian/projects/chrombert/chrombert_tools/ChromBERT-tools/examples/cli/output_find_driver_in_transition/acc/train/try_00_seed_55/lightning_logs/lightning_logs/version_0/checkpoints/epoch=2-step=163.ckpt" \
  --ft-ckpt-exp "/mnt/Storage2/home/chenqianqian/projects/chrombert/chrombert_tools/ChromBERT-tools/examples/cli/output_find_driver_in_transition/exp/train/try_00_seed_55/lightning_logs/lightning_logs/version_0/checkpoints/epoch=4-step=129.ckpt" \
  --genome 'hg38' \
  --resolution '1kb' \
  --odir output_find_driver_in_transition \
  --direction "2-1" 2> "./tmp/hg38_1kb.stderr.2.log"

Whether to train ChromBERT to predict chromatin accessibility changes in cell state transition: True
Stage 1 (acc): prepare dataset
Chromatin accessibility dataset already exists in output_find_driver_in_transition/acc/dataset
Finished Stage 1 (acc)
Stage 2 (acc): train ChromBERT to predict chromatin accessibility changes in cell state transition
Using fine-tuned checkpoint: /mnt/Storage2/home/chenqianqian/projects/chrombert/chrombert_tools/ChromBERT-tools/examples/cli/output_find_driver_in_transition/acc/train/try_00_seed_55/lightning_logs/lightning_logs/version_0/checkpoints/epoch=2-step=163.ckpt
Load pretrained ckpt /mnt/Storage/home/chenqianqian/.cache/chrombert/data/checkpoint/hg38_6k_1kb_pretrain.ckpt successfully!
Loading checkpoint from /mnt/Storage2/home/chenqianqian/projects/chrombert/chrombert_tools/ChromBERT-tools/examples/cli/output_find_driver_in_transition/acc/train/try_00_seed_55/lightning_logs/lightning_logs/version_0/checkpoints/epoch=2-step=163.ckpt
Loading from pl m

In [14]:
odir = "./output_find_driver_in_transition"
exp_odir = f'{odir}/exp'
exp_results_odir = f"{exp_odir}/results"
exp_rank_df = pd.read_csv(os.path.join(exp_results_odir, "factor_importance_rank.csv"))
exp_rank_df

,factors,similarity,rank,embedding_shift
0,histone lysine acetylation,0.921118,1,0.078882
1,histone lysine crotonylation,0.923516,2,0.076484
2,cbx6,0.929447,3,0.070553
3,cbx7,0.933499,4,0.066501
4,h2ak5ac,0.943805,5,0.056195
...,...,...,...,...
1067,snapc4,0.996417,1068,0.003583
1068,foxo4,0.996558,1069,0.003442
1069,id2,0.996619,1070,0.003381
1070,mafg,0.996837,1071,0.003163


In [15]:
acc_odir = f'{odir}/acc'
acc_results_odir = f"{acc_odir}/results"
acc_rank_df = pd.read_csv(os.path.join(acc_results_odir, "factor_importance_rank.csv"))
acc_rank_df

,factors,similarity,rank,embedding_shift
0,pax3-foxo1a,0.140829,1,0.859171
1,myog,0.150968,2,0.849032
2,myf5,0.181639,3,0.818361
3,myod1,0.267095,4,0.732905
4,pax7,0.304148,5,0.695852
...,...,...,...,...
1067,znf26,0.983500,1068,0.016500
1068,tox4,0.983787,1069,0.016213
1069,znf266,0.984463,1070,0.015537
1070,zbed4,0.984526,1071,0.015474


In [16]:
import pandas as pd
merge_odir=f'{odir}/merge'
merge_df = pd.read_csv(os.path.join(merge_odir, "factor_importance_rank.csv"))
merge_df.head(n=10)

,factors,similarity_exp,rank_exp,embedding_shift_exp,similarity_acc,rank_acc,embedding_shift_acc,total_rank
0,chd4,0.975052,25,0.024948,0.470624,15,0.529376,1
1,yap1,0.978078,34,0.021922,0.420340,13,0.579660,2
2,neurog2,0.980249,37,0.019751,0.402066,10,0.597934,2
3,myf5,0.983809,54,0.016191,0.181639,3,0.818361,4
4,h3k36me1,0.957156,8,0.042844,0.802063,57,0.197937,5
5,esco2,0.977604,33,0.022396,0.703099,34,0.296901,6
6,tcf21,0.982792,48,0.017208,0.538058,20,0.461942,7
7,pgbd3,0.984118,56,0.015882,0.418980,12,0.581020,7
8,tead1,0.985233,67,0.014767,0.357693,8,0.642307,9
9,myod1,0.986058,72,0.013942,0.267095,4,0.732905,10


In [17]:
merge_df = pd.merge(exp_rank_df,acc_rank_df,on='factors',how='inner',suffixes=['_exp','_acc'])
merge_df

,factors,similarity_exp,rank_exp,embedding_shift_exp,similarity_acc,rank_acc,embedding_shift_acc
0,histone lysine acetylation,0.921118,1,0.078882,0.937713,195,0.062287
1,histone lysine crotonylation,0.923516,2,0.076484,0.941648,217,0.058352
2,cbx6,0.929447,3,0.070553,0.894013,98,0.105987
3,cbx7,0.933499,4,0.066501,0.901252,108,0.098748
4,h2ak5ac,0.943805,5,0.056195,0.913511,130,0.086489
...,...,...,...,...,...,...,...
1067,snapc4,0.996417,1068,0.003583,0.971879,823,0.028121
1068,foxo4,0.996558,1069,0.003442,0.972842,844,0.027158
1069,id2,0.996619,1070,0.003381,0.972055,828,0.027945
1070,mafg,0.996837,1071,0.003163,0.978791,972,0.021209


In [18]:
merge_df['total_rank']=((merge_df['rank_exp']+merge_df['rank_acc'])/2).rank().astype(int)
merge_df = merge_df.sort_values('total_rank').reset_index(drop=True)
merge_df

,factors,similarity_exp,rank_exp,embedding_shift_exp,similarity_acc,rank_acc,embedding_shift_acc,total_rank
0,chd4,0.975052,25,0.024948,0.470624,15,0.529376,1
1,yap1,0.978078,34,0.021922,0.420340,13,0.579660,2
2,neurog2,0.980249,37,0.019751,0.402066,10,0.597934,2
3,myf5,0.983809,54,0.016191,0.181639,3,0.818361,4
4,h3k36me1,0.957156,8,0.042844,0.802063,57,0.197937,5
...,...,...,...,...,...,...,...,...
1067,hes4,0.996293,1064,0.003707,0.982166,1049,0.017834,1068
1068,znf26,0.996089,1047,0.003911,0.983500,1068,0.016500,1069
1069,zbtb10,0.996412,1067,0.003588,0.982231,1052,0.017769,1070
1070,zbed4,0.996220,1057,0.003780,0.984526,1071,0.015474,1071


### Load the fine-tuned checkpoint to infer key regulators and TRN for myoblast (skip fine-tuning)

In [19]:
# if you have already run find_driver_in_transition, you can use the fine-tuned checkpoint to infer drivers
ft_ckpt_exp_dir = "./output_find_driver_in_transition/exp/train/**/*.ckpt"
ft_ckpt_acc_dir = "./output_find_driver_in_transition/acc/train/**/*.ckpt"

import glob
ft_ckpt_exp = glob.glob(ft_ckpt_exp_dir, recursive=True)[0]
ft_ckpt_acc = glob.glob(ft_ckpt_acc_dir, recursive=True)[0]

In [20]:
ft_ckpt_exp,ft_ckpt_acc

('./output_find_driver_in_transition/exp/train/try_00_seed_55/lightning_logs/lightning_logs/version_0/checkpoints/epoch=4-step=129.ckpt',
 './output_find_driver_in_transition/acc/train/try_00_seed_55/lightning_logs/lightning_logs/version_0/checkpoints/epoch=2-step=163.ckpt')

In [23]:
!chrombert-tools predict_transition_driver_regulators \
  --ft-ckpt-exp {ft_ckpt_exp} \
  --ft-ckpt-acc {ft_ckpt_acc} \
  --genome 'hg38' \
  --exp-tpm1 "../data/fibroblast_expression_sample5000.csv" \
  --exp-tpm2 "../data/myoblast_expression_sample5000.csv" \
  --acc-peak1 "../data/fibroblast_ENCFF184KAM_peak.bed" \
  --acc-peak2 "../data/myoblast_ENCFF647RNC_peak.bed" \
  --acc-signal1 "../data/fibroblast_ENCFF361BTT_signal.bigwig" \
  --acc-signal2 "../data/myoblast_ENCFF149ERN_signal.bigwig" \
  --resolution '1kb' \
  --odir output_find_driver_in_transition_load_ckpt \
  --direction "2-1" 2> "./tmp/hg38_1kb.stderr.log"

Whether to train ChromBERT to predict chromatin accessibility changes in cell state transition: True
Stage 1 (acc): prepare dataset
Processing stage 1: prepare chromatin accessibility dataset
  Mode: two states (fold-change label)
  TSS ± flank background regions: enabled, flank distance: 10000 bp
[state1_0] Region summary - total: 287892, overlapping with ChromBERT: 284160 (one region may overlap multiple ChromBERT regions, we keep overlaps with ≥50% coverage of either the ChromBERT bin or the input region), non-overlapping: 5967
  [state1] merged 1 peak file(s) → 250143 unique ChromBERT regions (after overlap + dedup)
[state2_0] Region summary - total: 373422, overlapping with ChromBERT: 368260 (one region may overlap multiple ChromBERT regions, we keep overlaps with ≥50% coverage of either the ChromBERT bin or the input region), non-overlapping: 7920
  [state2] merged 1 peak file(s) → 324690 unique ChromBERT regions (after overlap + dedup)
[tss_bg] Region summary - total: 19260, ove

In [24]:
odir = "./output_find_driver_in_transition_load_ckpt"
exp_odir = f'{odir}/exp'
exp_results_odir = f"{exp_odir}/results"
exp_rank_df = pd.read_csv(os.path.join(exp_results_odir, "factor_importance_rank.csv"))
exp_rank_df.head(n=10)

,factors,similarity,rank,embedding_shift
0,histone lysine acetylation,0.921118,1,0.078882
1,histone lysine crotonylation,0.923516,2,0.076484
2,cbx6,0.929447,3,0.070553
3,cbx7,0.933499,4,0.066501
4,h2ak5ac,0.943805,5,0.056195
5,h2ax,0.947999,6,0.052001
6,h3k79me1,0.955875,7,0.044125
7,h3k36me1,0.957156,8,0.042844
8,cbx8,0.957597,9,0.042403
9,h2bk20ac,0.961643,10,0.038357


In [25]:
acc_odir = f'{odir}/acc'
acc_results_odir = f"{acc_odir}/results"
acc_rank_df = pd.read_csv(os.path.join(acc_results_odir, "factor_importance_rank.csv"))
acc_rank_df.head(n=10)

,factors,similarity,rank,embedding_shift
0,pax3-foxo1a,0.140829,1,0.859171
1,myog,0.150968,2,0.849032
2,myf5,0.181639,3,0.818361
3,myod1,0.267095,4,0.732905
4,pax7,0.304148,5,0.695852
5,ascl1,0.345684,6,0.654316
6,dux4,0.352310,7,0.647690
7,tead1,0.357693,8,0.642307
8,neurod1,0.387828,9,0.612172
9,neurog2,0.402066,10,0.597934


In [26]:
import pandas as pd
merge_odir=f'{odir}/merge'
merge_df = pd.read_csv(os.path.join(merge_odir, "factor_importance_rank.csv"))
merge_df.head(n=10)

,factors,similarity_exp,rank_exp,embedding_shift_exp,similarity_acc,rank_acc,embedding_shift_acc,total_rank
0,chd4,0.975052,25,0.024948,0.470624,15,0.529376,1
1,yap1,0.978078,34,0.021922,0.420340,13,0.579660,2
2,neurog2,0.980249,37,0.019751,0.402066,10,0.597934,2
3,myf5,0.983809,54,0.016191,0.181639,3,0.818361,4
4,h3k36me1,0.957156,8,0.042844,0.802063,57,0.197937,5
5,esco2,0.977604,33,0.022396,0.703099,34,0.296901,6
6,tcf21,0.982792,48,0.017208,0.538058,20,0.461942,7
7,pgbd3,0.984118,56,0.015882,0.418980,12,0.581020,7
8,tead1,0.985233,67,0.014767,0.357693,8,0.642307,9
9,myod1,0.986058,72,0.013942,0.267095,4,0.732905,10


In [27]:
merge_df = pd.merge(exp_rank_df,acc_rank_df,on='factors',how='inner',suffixes=['_exp','_acc'])
merge_df

,factors,similarity_exp,rank_exp,embedding_shift_exp,similarity_acc,rank_acc,embedding_shift_acc
0,histone lysine acetylation,0.921118,1,0.078882,0.937713,195,0.062287
1,histone lysine crotonylation,0.923516,2,0.076484,0.941648,217,0.058352
2,cbx6,0.929447,3,0.070553,0.894013,98,0.105987
3,cbx7,0.933499,4,0.066501,0.901252,108,0.098748
4,h2ak5ac,0.943805,5,0.056195,0.913511,130,0.086489
...,...,...,...,...,...,...,...
1067,snapc4,0.996417,1068,0.003583,0.971879,823,0.028121
1068,foxo4,0.996558,1069,0.003442,0.972842,844,0.027158
1069,id2,0.996619,1070,0.003381,0.972055,828,0.027945
1070,mafg,0.996837,1071,0.003163,0.978791,972,0.021209


In [28]:
merge_df['total_rank']=((merge_df['rank_exp']+merge_df['rank_acc'])/2).rank().astype(int)
merge_df = merge_df.sort_values('total_rank').reset_index(drop=True)
merge_df

,factors,similarity_exp,rank_exp,embedding_shift_exp,similarity_acc,rank_acc,embedding_shift_acc,total_rank
0,chd4,0.975052,25,0.024948,0.470624,15,0.529376,1
1,yap1,0.978078,34,0.021922,0.420340,13,0.579660,2
2,neurog2,0.980249,37,0.019751,0.402066,10,0.597934,2
3,myf5,0.983809,54,0.016191,0.181639,3,0.818361,4
4,h3k36me1,0.957156,8,0.042844,0.802063,57,0.197937,5
...,...,...,...,...,...,...,...,...
1067,hes4,0.996293,1064,0.003707,0.982166,1049,0.017834,1068
1068,znf26,0.996089,1047,0.003911,0.983500,1068,0.016500,1069
1069,zbtb10,0.996412,1067,0.003588,0.982231,1052,0.017769,1070
1070,zbed4,0.996220,1057,0.003780,0.984526,1071,0.015474,1071
